# Scalable & Resumable LLM Enrichment

This notebook implements a robust pipeline:
1.  **Load Base Data**: From `data/reviews_base/`.
2.  **Load Existing Progress**: From `data/reviews_enriched_v1/`.
3.  **Find Delta**: Identify rows that haven't been enriched yet (Anti-Join).
4.  **Process Batch**: Enrich a configurable chunk (e.g. 100 rows).
5.  **Append**: Save results, allowing the process to be stopped and resumed.

In [ ]:
# --- CONFIGURATION ---
import sys, os
from pathlib import Path
# Add src to sys.path
sys.path.append(os.path.abspath("../../../../.."))

from stampli.paths import REVIEWS_BASE_PARQUET, REVIEWS_ENRICHED_V1, REVIEWS_ENRICHED_V2_GEMMA, REVIEWS_ENRICHED_V3_LLAMA

BASE_PATH = str(REVIEWS_BASE_PARQUET)
ENRICHED_PATH = str(REVIEWS_ENRICHED_V1)
INGESTION_BATCH = 100  # Number of rows to process in this run

# LLM Configuration
LLM_PROVIDER = "openai" # Options: "openai", "ollama"
OLLAMA_MODEL = "llama3.2:3b-instruct-q4_0" # Best trade-off for JSON extraction
OPENAI_MODEL = "gpt-4o-mini"

In [ ]:
# Check available Ollama models if using Ollama
if LLM_PROVIDER == "ollama":
    print(f"Using Local Ollama with model: {OLLAMA_MODEL}")
    try:
        import subprocess
        res = subprocess.run(["ollama", "list"], capture_output=True, text=True)
        print("Available Models:\n", res.stdout)
    except Exception as e:
        print("Could not list ollama models (is ollama installed?):", e)
else:
    print(f"Using OpenAI with model: {OPENAI_MODEL}")

In [ ]:
import pandas as pd
import os

from stampli.util.display import display_scrollable_dataframe


In [ ]:
# 1. Setup Spark
try:
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import col, from_json, count
    from pyspark.sql.types import StructType, StructField, StringType, BooleanType, ArrayType, IntegerType
    from stampli.util.runtime import bootstrap_spark_env
except ImportError:
    print("Please install pyspark: pip install pyspark")

bootstrap_spark_env()

spark = (SparkSession.builder
    .appName("StampliEnrichmentResumable")
    .config("spark.executor.memory", "16g")
    .config("spark.driver.memory", "16g")
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")
print("Spark Session Created (16GB)")

In [ ]:
# 2. Load Base Data
if not os.path.exists(BASE_PATH):
    print(f"Error: Base path {BASE_PATH} does not exist. Run ingest_spark.ipynb first.")
else:
    sdf_base = spark.read.parquet(BASE_PATH)
    print(f"Base Rows: {sdf_base.count()}")

In [ ]:
# 3. Load Prior Enrichment (Handle First Run)
import os

if os.path.exists(ENRICHED_PATH) and len(os.listdir(ENRICHED_PATH)) > 0:
    try: 
        sdf_done = spark.read.parquet(ENRICHED_PATH)
        done_count = sdf_done.count()
        print(f"Already Enriched: {done_count}")
    except Exception:
        # Possible directory exists but is empty or corrupt
        print("Enriched directory empty or invalid. Starting fresh.")
        sdf_done = None
        done_count = 0
else:
    print("No prior enrichment found. Starting fresh.")
    sdf_done = None
    done_count = 0

In [ ]:
# 4. Calculate To-Do List (Left Anti Join)
if sdf_done is not None:
    # Exclude rows where review_uid is already in df_done
    sdf_todo = sdf_base.join(sdf_done, "review_uid", "left_anti")
else:
    sdf_todo = sdf_base

todo_count = sdf_todo.count()
print(f"Remaining To Do: {todo_count}")

if todo_count == 0:
    print("All done!")
else:
    # 5. Limit Batch
    df_batch = sdf_todo.limit(INGESTION_BATCH)
    print(f"Processing Batch of: {INGESTION_BATCH}")

In [ ]:
import json
import os
from typing import Iterator
import pandas as pd

# ----------------------------
# 1. Output Schema (Spark)
# ----------------------------
ENRICHED_SCHEMA = StructType([
    # Core Enriched Fields
    StructField("sentiment_score", IntegerType(), True),
    StructField("sentiment_label", StringType(), True),
    StructField("topics", ArrayType(StringType()), True),
    StructField("is_complaint", BooleanType(), True),
    
    # High-Signal Extras
    StructField("crowd_level", StringType(), True),
    StructField("queue_time_rating", StringType(), True),
    StructField("staff_sentiment", StringType(), True),
    StructField("price_sensitivity", StringType(), True),
    StructField("family_sentiment", StringType(), True),
    StructField("summary", StringType(), True),
    StructField("entities", ArrayType(StringType()), True)
])

# Intermediate Schema (Passthrough + JSON)
BATCH_SCHEMA = StructType([
    StructField("review_uid", StringType(), True),
    StructField("Review_Text", StringType(), True),
    
    # Metadata Passthrough
    StructField("Rating", IntegerType(), True),
    StructField("Reviewer_Location", StringType(), True),
    StructField("Year_Month", StringType(), True),
    StructField("Branch", StringType(), True),
    
    # New Payload
    StructField("enriched_json", StringType(), True)
])

# Pass configuration as default args to avoid serialization issues for simple globals
def enrich_partition(iterator: Iterator[pd.DataFrame], provider=LLM_PROVIDER, ol_model=OLLAMA_MODEL, oa_model=OPENAI_MODEL) -> Iterator[pd.DataFrame]:
    from langchain_openai import ChatOpenAI
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.output_parsers import PydanticOutputParser
    from pydantic import BaseModel, Field
    from typing import List, Optional

    # ----------------------------
    # 2. Pydantic Model (LLM)
    # ----------------------------
    class ReviewEnrichment(BaseModel):
        sentiment_score: int = Field(..., description="1-5 score")
        sentiment_label: str = Field(..., description="Positive/Negative/Neutral")
        topics: List[str] = Field(..., description="List of main topics")
        is_complaint: bool = Field(..., description="True if complaint")
        
        # High-Signal Extras
        crowd_level: Optional[str] = Field(None, description="Enum: Empty, Moderate, Packed")
        queue_time_rating: Optional[str] = Field(None, description="Enum: Acceptable, Long, Unbearable")
        staff_sentiment: Optional[str] = Field(None, description="Enum: Friendly, Rude, Neutral")
        price_sensitivity: Optional[str] = Field(None, description="Enum: Worth it, Expensive, Rip-off")
        family_sentiment: Optional[str] = Field(None, description="Enum: Harmonious, Joyful, Neutral, Tired, Stressed, Conflict")
        summary: Optional[str] = Field(None, description="1-sentence dense summary")
        entities: Optional[List[str]] = Field(None, description="Named entities mentioned")

    if provider == "ollama":
        # Use OpenAI client pointing to Ollama endpoint
        llm = ChatOpenAI(
            base_url="http://localhost:11434/v1",
            api_key="ollama",
            model=ol_model,
            temperature=0
        )
    else:
        api_key = os.environ.get("OPENAI_API_KEY")
        llm = ChatOpenAI(model=oa_model, api_key=api_key, temperature=0)

    parser = PydanticOutputParser(pydantic_object=ReviewEnrichment)
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a data extraction engine. Return VALID JSON ONLY. No markdown, no thoughts.\n{format_instructions}"),
        ("user", "{text}")
    ])
    chain = prompt | llm | parser
    
    for pdf in iterator:
        results = []
        format_instr = parser.get_format_instructions()
        for text in pdf["Review_Text"]:
            try:
                obj = chain.invoke({"text": text, "format_instructions": format_instr})
                results.append(obj.model_dump_json())
            except Exception:
                results.append(json.dumps({}))
        
        pdf["enriched_json"] = results
        # Return ALL columns (Review_Text, review_uid + Metadata + JSON)
        yield pdf


# 6. Run Enrichment (Action)
if todo_count > 0:
    # Ensure we select metadata columns for passthrough
    target_cols = ["review_uid", "Review_Text", "Rating", "Reviewer_Location", "Year_Month", "Branch"]
    
    raw_enriched = df_batch.select(*target_cols).mapInPandas(enrich_partition, schema=BATCH_SCHEMA)
    
    # 7. Parse & Expand
    final_sdf = raw_enriched \
        .withColumn("parsed", from_json(col("enriched_json"), ENRICHED_SCHEMA)) \
        .select(*target_cols, "enriched_json", "parsed.*") # Select Metadata + Raw JSON + Expanded Fields
    
    # 8. Append to Disk (Single Action)
    print(f"Writing batch of {INGESTION_BATCH} to {ENRICHED_PATH}...")
    final_sdf.write.mode("append").parquet(ENRICHED_PATH)
    print("Batch Complete.")

In [ ]:
# sanity read enriched into pandas dataframe

pd_sanity = pd.read_parquet(ENRICHED_PATH) 

len(pd_sanity)

In [ ]:
display_scrollable_dataframe(pd_sanity)

In [ ]:
# %% [markdown]
# ## 5. Validate Gemma 2 (V2) Output

ENRICHED_V2_PATH = str(REVIEWS_ENRICHED_V2_GEMMA)
if os.path.exists(ENRICHED_V2_PATH):
    print(f"Reading V2 Data from: {ENRICHED_V2_PATH}")
    df_v2 = pd.read_parquet(ENRICHED_V2_PATH)
    display(df_v2.head(10))
    print(f"V2 Count: {len(df_v2)}")
else:
    print("V2 Data not found. Run spark_enrichment_v2_gemma.py first.")

In [ ]:
# %% [markdown]
# ## 6. Validate Llama 3.1 (V3) Output

ENRICHED_V3_PATH = str(REVIEWS_ENRICHED_V3_LLAMA)
if os.path.exists(ENRICHED_V3_PATH):
    print(f"Reading V3 Data from: {ENRICHED_V3_PATH}")
    df_v3 = pd.read_parquet(ENRICHED_V3_PATH)
    display(df_v3.head(10))
    print(f"V3 Count: {len(df_v3)}")
else:
    print("V3 Data not found. Run spark_enrichment_v3_llama.py first.")